# Stripping in the accelerator

An H$^-$ ion holds its second electron by 0.75 eV, and any collision on the way out of the
source can take it. The gas that leaks from the source through the grid apertures fills the
accelerator, and an H$^-$ stripped there continues as a fast atom: with the energy it had
gained where it was stripped, in the direction it had there, no longer accelerated or
focused. In a negative-ion injector this is the main loss of the accelerator.

This example takes the round H$^-$ beamlet of `beamtracer.examples.negative_ion_extraction_optics`
(plasma grid at 0 V, extraction grid at 5 kV, grounded grid at 20 kV) and adds the gas
between its grids. First every trajectory carries the fractions of its particles in each
charge state (`mode="fractions"`); then every trajectory is in one state and changes it at
sampled events (`mode="analog"`), which shows where the stripped atoms go.

In [ ]:
from dataclasses import replace

import numpy as np
import plasma_plots as pp
from plasmacoll import charge_state_fractions
from plasmacoll.beam_gas import hydrogen_beam_on_h2
from scipy import constants

from beamtracer import BeamOptics, ChargeChangingParams
from beamtracer.examples.negative_ion_extraction_optics import (
    H_MINUS_MASS,
    BeamletCase,
    beamlet_config,
)
from beamtracer.solver.charge_changing import AxialProfile, conductance_profile

BLUE, ORANGE, GREEN, GREY = "C0", "C1", "C2", "0.6"
pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

## The gas

The source is filled with H$_2$ at about 0.3 Pa, $7.2 \times 10^{19}$ m$^{-3}$ at room
temperature; behind the accelerator the pumps keep it near $10^{18}$ m$^{-3}$. In between,
`conductance_profile` gives the density of a free-molecular flow through the three grid
apertures, each a short tube with Clausing's transmission probability, and uniform gas in the
gaps. Most of the drop is across the narrowest aperture, that of the extraction grid.

The co-extracted electrons of the example are left out: the cross sections below are those of
the H$^-$ beam.

In [ ]:
case = BeamletCase(electron_ratio=0.0)
grids = {
    "PG": (*case.pg, case.pg_radius),
    "EG": (*case.eg, case.eg_radius),
    "GG": (*case.gg, case.gg_radius),
}  # (front, back, aperture radius) [m]
gas = conductance_profile(list(grids.values()), upstream=7.2e19, downstream=1.0e18)
profile = AxialProfile(gas.x, gas.density)

positions = np.linspace(0.0, case.length, 400)  # [m]
with pp.figure(title="H$_2$ density along the accelerator") as fig:
    ax = fig[0]
    ax.plot(positions * 1e3, profile(positions), color=BLUE)
    for name, (front, back, _) in grids.items():
        ax.axvspan(front * 1e3, back * 1e3, color=GREY, alpha=0.4)
        ax.text((front + back) / 2 * 1e3, 7.5e19, name, ha="center")
    ax.set_xlabel("$x$ [mm]  (grey: the grids)")
    ax.set_ylabel("$n_{H_2}$ [m$^{-3}$]")

## The cross sections

`plasmacoll.beam_gas.hydrogen_beam_on_h2` holds the reactions of a fast hydrogen beam with
H$_2$. The model takes one cross section per transition, so they are taken at 10 keV, the
middle of the H$^-$ energies in this accelerator; H$^- \to$ H is Janev, Reiter and Samm's
data, the double stripping H$^- \to$ H$^+$ and the ionization of the atoms H $\to$ H$^+$ are
placeholders of the right order (plasmacoll says which).

In [ ]:
reactions = hydrogen_beam_on_h2()
transitions = reactions.transitions(10.0e3)
sigma = {}
for (start, end), value in transitions.cross_sections["H2"].items():
    sigma.setdefault(start, {})[end] = value
for start, row in sigma.items():
    for end, value in row.items():
        print(f"{start:>2} -> {end:<2} {value:.2e} m^2")
print("placeholders:", ", ".join(reactions.placeholders))

## Fractions of each trajectory

The beamlet solved with the charge changes on the gas profile, and twelve diagnostic planes
along the axis. In the fractions mode the force on a trajectory is that of the mean charge of
its particles, the space charge that of every state with its sign.

In [ ]:
planes = list(np.linspace(1.0e-3, case.exit_plane, 12))
targets = {"H2": {"x": gas.x, "density": gas.density}}
config = beamlet_config(case).replace(
    charge_changing=ChargeChangingParams({"H2": sigma}, targets),
    diagnostic_planes=planes,
    mixing="anderson",
)
model = BeamOptics(config).run()
change = model.state.history[-1]["max_potential_change"]
print(f"{model.stats['iterations']} iterations, last potential change {change:.2f} V")


def shares(run, x):
    # the share of the beam's current in each charge state at the plane x
    total = run.beam_moments(x).current
    return np.array([run.state_moments(x, s).current / total for s in "-0+"])


exit_shares = shares(model, case.exit_plane)
print("at the exit: H- {:.4f}, H {:.4f}, H+ {:.4f}".format(*exit_shares))

The iteration stops at its limit of 40 iterations with the potential still changing by a
volt or two out of 20 kV, from trajectories near the edge of the meniscus; the stripping, a
few per cent of the beam, does not notice.

The fractions along the beam against those of the line density along the axis,
$\int_0^x n_{H_2}\,dx'$, the estimate of a beam that goes straight: almost all of the
stripping happens before the extraction grid, where the gas is densest.

In [ ]:
along = np.array([shares(model, x) for x in planes])
axis = np.array(
    [
        charge_state_fractions(
            transitions, {"H2": [0.0, profile.line_density(0.0, x)]}
        )[-1]
        for x in positions
    ]
)
with pp.figure(title="Charge states along the accelerator") as fig:
    ax = fig[0]
    for k, (label, color) in enumerate(
        (("H$^-$ lost", BLUE), ("H", ORANGE), ("H$^+$", GREEN))
    ):
        traced = 1.0 - along[:, 0] if k == 0 else along[:, k]
        estimate = 1.0 - axis[:, 0] if k == 0 else axis[:, k]
        ax.plot(positions * 1e3, estimate, color=color, ls="--", lw=1)
        ax.plot(np.array(planes) * 1e3, traced, "o", color=color, label=label)
    for front, back, _ in grids.values():
        ax.axvspan(front * 1e3, back * 1e3, color=GREY, alpha=0.4)
    ax.set_xlabel("$x$ [mm]  (dashed: along the axis)")
    ax.set_ylabel("share of the beam")
    ax.legend()

## Trajectories that change state

The same beamlet with `mode="analog"`: every trajectory is in one state and changes it where
an optical depth drawn from an exponential distribution runs out, with the force of its actual
charge from there on. `splitting=8` splits a trajectory at its first event into eight of an
eighth of its current, which draw their next states independently: more atoms for the same
number of trajectories launched. The mean over the trajectories is the fractions mode.

In [ ]:
analog = BeamOptics(
    config.replace(
        beams=[replace(config.beams[0], num_particles=1000)],
        charge_changing=ChargeChangingParams(
            {"H2": sigma}, targets, mode="analog", splitting=8, split_events=1, seed=1
        ),
        diagnostic_planes=[case.exit_plane],
        save_trajectories=0,
    )
).run()
found = shares(analog, case.exit_plane)
print(
    f"{analog.stats['launched']} trajectories, {analog.stats['launched'] - 1000} of them split off"
)
print("at the exit: H- {:.4f}, H {:.4f}, H+ {:.4f}".format(*found))
print("fractions mode: H- {:.4f}, H {:.4f}, H+ {:.4f}".format(*exit_shares))

The analog mode gives the fractions mode's shares within its sampling noise. What it adds is
the atoms themselves. An H$^-$ crosses the grounded grid with the full 20 keV; an atom keeps
the energy of the place where it was stripped, so the atoms leave with every energy between,
half of them with less than a quarter of the full energy, since most are stripped before the
extraction grid.

In [ ]:
hits = analog.plane(case.exit_plane)
mass = H_MINUS_MASS * constants.atomic_mass
energy = 0.5 * mass * np.sum(hits.velocity**2, axis=0) / constants.e  # eV
atom = hits.fractions[1] == 1.0
ion = hits.fractions[0] == 1.0
order = np.argsort(energy[atom])
weight = np.cumsum(hits.current[atom][order]) / hits.current[atom].sum()
median = energy[atom][order][np.searchsorted(weight, 0.5)]
print(
    f"H- leave with {np.mean(energy[ion]) / 1e3:.1f} keV, half of the atoms with less than {median / 1e3:.1f} keV"
)
with pp.figure(title="Energy of the atoms at the exit") as fig:
    ax = fig[0]
    ax.hist(
        energy[atom] / 1e3,
        bins=np.linspace(0.0, 21.0, 22),  # 1 keV
        weights=hits.current[atom] / hits.current.sum(),
        color=ORANGE,
    )
    ax.axvline(np.mean(energy[ion]) / 1e3, color=BLUE, ls="--", label="H$^-$")
    ax.set_xlabel("kinetic energy [keV]")
    ax.set_ylabel("share of the beam")
    ax.legend()

The atoms are not focused by the rest of the accelerator either: they leave with the angle
they had where they were stripped, in the diverging field of the meniscus and the extraction
gap.

In [ ]:
for state, name in (("-", "H-"), ("0", "H")):
    moments = analog.state_moments(case.exit_plane, state)
    print(
        f"{name:3s} {moments.current / hits.current.sum():6.2%} of the beam, "
        f"rms angle {moments.rms_angle * 1e3:5.1f} mrad"
    )

About 6 % of the H$^-$ is stripped in this accelerator, nearly all of it upstream of the
extraction grid, and the atoms leave with a fraction of the energy and about three times the
divergence of the beam: power that the beamline downstream sees as a halo. A longer or
multi-stage accelerator strips more, at higher energies. The cross sections are taken at one
energy, and the gas does not see the beam: for its density from a neutral-transport run,
`densities` takes the profile of any `{"x": ..., "density": ...}`.